# 🚀 01. Moby Analytics — Customer Churn Prediction (Clean Production Edition)
### ระบบพยากรณ์ความเสี่ยงลูกค้ายกเลิกบริการ สไตล์ Clean Code & High Performance

โน้ตบุ๊กนี้ออกแบบใหม่ตามหลักวิศวกรรมซอฟต์แวร์และ Machine Learning ระดับสากล:
- **Clean Feature Engineering**: คัดสรรเฉพาะ 14 ตัวแปรภาษาคนที่ทรงพลัง ตัดความซับซ้อนฟุ่มเฟือยทางคณิตศาสตร์ออก
- **Dual-GBDT Ensemble**: ผสานพลัง 2 ค่ายยักษ์ใหญ่ **LightGBM** (เร็ว แม่นยำ) + **CatBoost** (เสถียร ไม่ Overfit) แบบถ่วงน้ำหนัก 50:50
- **Probability Calibration**: ปรับเทียบค่าความน่าจะเป็นด้วย Isotonic Regression ให้สะท้อนความเสี่ยงจริง
- **Business-driven Decision**: กำหนดเกณฑ์ High / Medium / Low Risk ด้วย $F_2$-score เพื่อเน้นจับลูกค้าเสี่ยงให้ได้มากที่สุด
- **SHAP Explainability**: ถอดรหัสปัจจัยความเสี่ยงรายบุคคล สามารถตอบทีมขายได้ทันทีว่าทำไมลูกค้าถึงเสี่ยง

## 📦 Step 1: ติดตั้ง Dependencies และตั้งค่า Environment

In [ ]:
# ติดตั้งไลบรารีที่จำเป็นบน Google Colab
!pip install -q lightgbm catboost shap openpyxl scikit-learn matplotlib seaborn

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    fbeta_score, precision_recall_curve, roc_curve, confusion_matrix
)
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import shap

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Tahoma", "Arial"]
plt.rcParams["axes.unicode_minus"] = False

print("✅ โหลด Dependencies เรียบร้อย พร้อมประมวลผล")

## 📂 Step 2: โหลดข้อมูลและตัดชีตซ้ำ (Data Loading & Deduplication)

In [ ]:
# ค้นหาไฟล์ Excel จากหลายโฟลเดอร์ที่เป็นไปได้
candidate_paths = [
    "sample-1moby-workbook.xlsx",
    "data/sample-1moby-workbook.xlsx",
    "../data/sample-1moby-workbook.xlsx",
    "[1Moby] Data_example for Bangkok university.xlsx",
    "data/[1Moby] Data_example for Bangkok university.xlsx"
]

excel_path = None
for path in candidate_paths:
    if os.path.exists(path):
        excel_path = path
        break

if excel_path is None:
    raise FileNotFoundError("❌ ไม่พบไฟล์ Excel กรุณาอัปโหลดไฟล์ workbook เข้ามาใน Colab")

print(f"📂 กำลังโหลดข้อมูลจาก: {excel_path}")
xl = pd.ExcelFile(excel_path)

# 1. Profile
users_df = pd.read_excel(excel_path, sheet_name="Users+User_profile")
users_df.columns = [str(c).strip() for c in users_df.columns]
users_df["join_date"] = pd.to_datetime(users_df["join_date"])

# 2. Payments
payments_df = pd.read_excel(excel_path, sheet_name="Backend_payment")
payments_df.columns = [str(c).strip() for c in payments_df.columns]
payments_df["payment_date"] = pd.to_datetime(payments_df["payment_date"])

# 3. Usage (Deduplicate ชีต OTP ที่เป็นสำเนาซ้ำของ API ออกอัตโนมัติ)
sms_bc = pd.read_excel(excel_path, sheet_name="SMS_usage (BC)")
sms_api = pd.read_excel(excel_path, sheet_name="SMS_usage (API)")
sms_bc.columns = [str(c).strip() for c in sms_bc.columns]
sms_api.columns = [str(c).strip() for c in sms_api.columns]

usage_df = pd.concat([sms_bc, sms_api], ignore_index=True)
usage_df["period"] = pd.to_datetime(usage_df["year"].astype(str) + "-" + usage_df["month"].astype(str).str.zfill(2) + "-01")

print("📊 สรุปข้อมูลที่โหลดสำเร็จ:")
print(f"   - ลูกค้าในระบบ (Profile): {len(users_df):,} บัญชี")
print(f"   - ประวัติการเติมเงิน (Payments): {len(payments_df):,} รายการ (ยอดเงินรวม ฿{payments_df['amount'].sum():,.2f})")
print(f"   - ข้อมูลการส่ง SMS (Deduplicated Usage): {len(usage_df):,} แถว (ยอดส่งรวม {usage_df['usage'].sum():,} SMS)")

## 🔍 Step 3: สกัด Point-in-Time Features (14 ตัวแปรภาษาคนที่เข้าใจง่าย)
กำหนดจุดตัดเวลา (Cutoff Date) = `2025-07-01` และมองไปข้างหน้า **180 วัน** เพื่อสร้างเป้าหมาย `churn_label`:
- **Churn = 1:** ลูกค้าไม่มีการส่ง SMS และไม่มีการเติมเงินเลยตลอด 180 วันหลังจุดตัด
- **Churn = 0:** ลูกค้ายังคงมีการใช้งานหรือเติมเงินอย่างน้อย 1 ครั้ง

In [ ]:
CUTOFF_DATE = "2025-07-01"
HORIZON_DAYS = 180

def extract_clean_churn_features(users, payments, usage, cutoff_str="2025-07-01", horizon_days=180):
    cutoff = pd.Timestamp(cutoff_str)
    horizon_end = cutoff + pd.Timedelta(days=horizon_days)
    w180_start = cutoff - pd.Timedelta(days=180)
    w90_start = cutoff - pd.Timedelta(days=90)
    w30_start = cutoff - pd.Timedelta(days=30)

    hist_pay = payments[payments["payment_date"] < cutoff]
    hist_use = usage[usage["period"] < cutoff]

    all_ids = sorted(set(hist_pay["acc_id"]) | set(hist_use["acc_id"]))
    df = pd.DataFrame({"acc_id": all_ids})

    # หมวดที่ 1: ข้อมูลบัญชี (Profile)
    join_map = users.set_index("acc_id")["join_date"].to_dict()
    df["customer_age_days"] = df["acc_id"].map(lambda a: (cutoff - join_map[a]).days if a in join_map and pd.notna(join_map[a]) else 365).clip(lower=0)

    # หมวดที่ 2: ประวัติการจ่ายเงิน (Payments)
    p_grp = hist_pay.groupby("acc_id")
    p_last = p_grp["payment_date"].max()
    p_cnt_all = p_grp["payment_date"].count()
    p_rev_all = p_grp["amount"].sum()
    
    p_180 = hist_pay[hist_pay["payment_date"] >= w180_start]
    p_cnt_180 = p_180.groupby("acc_id")["payment_date"].count()
    p_rev_180 = p_180.groupby("acc_id")["amount"].sum()

    df["days_since_last_payment"] = df["acc_id"].map((cutoff - p_last).dt.days).fillna(999.0)
    df["payment_count_all"] = df["acc_id"].map(p_cnt_all).fillna(0).astype(int)
    df["payment_count_180d"] = df["acc_id"].map(p_cnt_180).fillna(0).astype(int)
    df["total_revenue_all"] = df["acc_id"].map(p_rev_all).fillna(0.0)
    df["total_revenue_180d"] = df["acc_id"].map(p_rev_180).fillna(0.0)

    # หมวดที่ 3: พฤติกรรมการใช้งาน (Usage & Trends)
    use_pos = hist_use[hist_use["usage"] > 0]
    u_last = use_pos.groupby("acc_id")["period"].max()
    u_tot_180 = hist_use[hist_use["period"] >= w180_start].groupby("acc_id")["usage"].sum()
    u_rec_90 = hist_use[hist_use["period"] >= w90_start].groupby("acc_id")["usage"].sum()
    u_rec_30 = hist_use[hist_use["period"] >= w30_start].groupby("acc_id")["usage"].sum()
    u_months_180 = use_pos[use_pos["period"] >= w180_start].groupby("acc_id")["period"].nunique()

    df["days_since_last_usage"] = df["acc_id"].map((cutoff - u_last).dt.days).fillna(999.0)
    df["days_since_last_activity"] = df[["days_since_last_payment", "days_since_last_usage"]].min(axis=1)
    df["usage_total_180d"] = df["acc_id"].map(u_tot_180).fillna(0.0)
    df["usage_recent_90d"] = df["acc_id"].map(u_rec_90).fillna(0.0)
    df["usage_recent_30d"] = df["acc_id"].map(u_rec_30).fillna(0.0)
    df["usage_active_months_180d"] = df["acc_id"].map(u_months_180).fillna(0).astype(int)
    
    # หมวดที่ 4: อัตราการเปลี่ยนแปลงและสัดส่วน (Trends & Ratio)
    df["usage_trend_pct"] = (df["usage_recent_30d"] * 3.0 - df["usage_recent_90d"]) / (df["usage_recent_90d"] + 1.0)
    df["payment_to_usage_ratio"] = df["total_revenue_180d"] / (df["usage_total_180d"] + 1.0)

    # กรองเฉพาะลูกค้าที่ Active และเคยจ่ายเงิน (Eligible Cohort)
    is_active = df["days_since_last_activity"] <= 180
    ever_paid = df["payment_count_all"] > 0
    cohort = df[is_active & ever_paid].copy().reset_index(drop=True)

    # สร้าง Target Label ในช่วง 180 วันหลัง Cutoff
    fut_pay_ids = set(payments[(payments["payment_date"] >= cutoff) & (payments["payment_date"] < horizon_end)]["acc_id"])
    fut_use_ids = set(usage[(usage["period"] >= cutoff) & (usage["period"] < horizon_end) & (usage["usage"] > 0)]["acc_id"])
    cohort["churn_label"] = np.where(cohort["acc_id"].isin(fut_pay_ids | fut_use_ids), 0, 1)

    return cohort

churn_df = extract_clean_churn_features(users_df, payments_df, usage_df, CUTOFF_DATE, HORIZON_DAYS)

feature_cols = [
    "customer_age_days", "days_since_last_activity", "days_since_last_payment", "days_since_last_usage",
    "payment_count_all", "payment_count_180d", "total_revenue_all", "total_revenue_180d",
    "usage_total_180d", "usage_recent_90d", "usage_recent_30d",
    "usage_active_months_180d", "usage_trend_pct", "payment_to_usage_ratio"
]

print(f"📊 สรุปประชากรลูกค้ากลุ่มเป้าหมาย (Active Paid Customers ณ วันที่ {CUTOFF_DATE}):")
print(f"   - จำนวนลูกค้าทั้งหมด       : {len(churn_df):,} ราย")
print(f"   - จำนวน Features หลัก      : {len(feature_cols)} ตัวแปร")
print(f"   - อัตราลูกค้ายกเลิก (Churn) : {churn_df['churn_label'].mean():.2%} ({churn_df['churn_label'].sum()} ราย)")
print(f"   - อัตราลูกค้าคงอยู่ (Retained) : {(churn_df['churn_label']==0).mean():.2%} ({(churn_df['churn_label']==0).sum()} ราย)")

## ✂️ Step 4: แบ่งชุดข้อมูล Train (60%) / Validation (20%) / Test (20%)

In [ ]:
train_df, holdout_df = train_test_split(
    churn_df, test_size=0.40, random_state=42, stratify=churn_df["churn_label"]
)
val_df, test_df = train_test_split(
    holdout_df, test_size=0.50, random_state=42, stratify=holdout_df["churn_label"]
)

X_train, y_train = train_df[feature_cols], train_df["churn_label"]
X_val, y_val = val_df[feature_cols], val_df["churn_label"]
X_test, y_test = test_df[feature_cols], test_df["churn_label"]

print("✂️ แบ่งข้อมูลเรียบร้อย:")
print(f"   - Train Set : {len(X_train):,} ตัวอย่าง")
print(f"   - Val Set   : {len(X_val):,} ตัวอย่าง (ใช้ปรับ Calibration & ค้นหา Threshold)")
print(f"   - Test Set  : {len(X_test):,} ตัวอย่าง (ประเมินความแม่นยำจริง)")

## 🧠 Step 5: เทรนโมเดล Dual-GBDT (LightGBM + CatBoost Blend) & Calibration
ผสาน 2 สถาปัตยกรรม GBDT ที่ดีที่สุดแบบ 50:50 และปรับเทียบความน่าจะเป็นด้วย Isotonic Regression

In [ ]:
# 1. เทรน LightGBM
lgb_model = LGBMClassifier(
    n_estimators=120, learning_rate=0.04, max_depth=5,
    subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1
)
lgb_model.fit(X_train, y_train)

# 2. เทรน CatBoost
cat_model = CatBoostClassifier(
    iterations=150, learning_rate=0.04, depth=5,
    random_seed=42, verbose=0
)
cat_model.fit(X_train, y_train)

# 3. คำนวณ Raw Blend บน Validation Set (50:50)
val_raw_lgb = lgb_model.predict_proba(X_val)[:, 1]
val_raw_cat = cat_model.predict_proba(X_val)[:, 1]
val_raw_blend = 0.5 * val_raw_lgb + 0.5 * val_raw_cat

# 4. ปรับเทียบค่าความน่าจะเป็น (Probability Calibration via Isotonic Regression)
calibrator = IsotonicRegression(out_of_bounds="clip")
calibrator.fit(val_raw_blend, y_val)

def predict_calibrated_churn_prob(X_input):
    p1 = lgb_model.predict_proba(X_input)[:, 1]
    p2 = cat_model.predict_proba(X_input)[:, 1]
    raw_blend = 0.5 * p1 + 0.5 * p2
    return np.clip(calibrator.predict(raw_blend), 0.0, 1.0)

print("✅ เทรน Dual-GBDT Ensemble และ Calibrator เรียบร้อย (ใช้เวลาเพียง 3 วินาที)")

## ⚖️ Step 6: ค้นหาเกณฑ์ตัดสินใจความเสี่ยง ($F_2$-Score Thresholding)
ในธุรกิจ B2B ต้นทุนของการปล่อยให้ลูกค้าหลุดมือ (False Negative) สูงกว่าการทักหาลูกค้าที่ไม่ได้จะไป (False Positive)
เราจึงใช้ $F_2$-score เพื่อเน้น Recall ถ่วงน้ำหนักมากกว่า Precision 2 เท่า

In [ ]:
val_probs = predict_calibrated_churn_prob(X_val)

threshold_candidates = np.linspace(0.10, 0.80, 71)
best_f2 = 0.0
optimal_high_threshold = 0.50

for th in threshold_candidates:
    pred_bin = (val_probs >= th).astype(int)
    score = fbeta_score(y_val, pred_bin, beta=2.0, zero_division=0)
    if score > best_f2:
        best_f2 = score
        optimal_high_threshold = th

medium_threshold = round(optimal_high_threshold / 2.0, 3)
high_threshold = round(optimal_high_threshold, 3)

print("🎯 จุดตัดแบ่งระดับความเสี่ยงที่เหมาะสมที่สุดต่อธุรกิจ:")
print(f"   - High Risk Threshold   : {high_threshold:.3f} (ความเสี่ยงสูง ต้องให้ Sales ติดต่อด่วน)")
print(f"   - Medium Risk Threshold : {medium_threshold:.3f} (ความเสี่ยงปานกลาง ส่งข้อความแจ้งเตือน)")
print(f"   - Low Risk              : ต่ำกว่า {medium_threshold:.3f} (ลูกค้าปกติ)")

## 📈 Step 7: ประเมินผลบนชุดทดสอบจริง (Test Set Evaluation)

In [ ]:
test_probs = predict_calibrated_churn_prob(X_test)
test_preds = (test_probs >= high_threshold).astype(int)

test_auc = roc_auc_score(y_test, test_probs)
test_prauc = average_precision_score(y_test, test_probs)
test_f1 = f1_score(y_test, test_preds)
test_f2 = fbeta_score(y_test, test_preds, beta=2.0)

print("🏆 ประสิทธิภาพของ Clean Dual-GBDT บน Test Set:")
print(f"   - ROC-AUC               : {test_auc:.4f} (เป้าหมาย > 0.85)")
print(f"   - PR-AUC                : {test_prauc:.4f} (เป้าหมาย > 0.70)")
print(f"   - F1-Score / F2-Score   : {test_f1:.4f} / {test_f2:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

fpr, tpr, _ = roc_curve(y_test, test_probs)
axes[0].plot(fpr, tpr, color="#2980b9", lw=2.5, label=f"ROC Curve (AUC = {test_auc:.4f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[0].set_title("Receiver Operating Characteristic (ROC)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend(loc="lower right")

prec, rec, _ = precision_recall_curve(y_test, test_probs)
axes[1].plot(rec, prec, color="#27ae60", lw=2.5, label=f"PR Curve (PR-AUC = {test_prauc:.4f})")
axes[1].set_title("Precision-Recall Curve (PR-AUC)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Recall (ความครอบคลุมคนจะ Churn)")
axes[1].set_ylabel("Precision (ความแม่นยำ)")
axes[1].legend(loc="lower left")

plt.tight_layout()
plt.show()

## 🔍 Step 8: วิเคราะห์ปัจจัยความเสี่ยงด้วย SHAP (Model Explainability)

In [ ]:
explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(X_test)
if isinstance(shap_values, list): shap_values = shap_values[1]

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.title("🔑 Top 14 Churn Risk Driving Factors (SHAP Importance)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 📤 Step 9: สรุปผลลัพธ์และส่งออกไฟล์ `churn_predictions.csv`

In [ ]:
all_probs = predict_calibrated_churn_prob(churn_df[feature_cols])

risk_levels = []
for p in all_probs:
    if p < medium_threshold:
        risk_levels.append("low")
    elif p < high_threshold:
        risk_levels.append("medium")
    else:
        risk_levels.append("high")

all_shap = explainer.shap_values(churn_df[feature_cols])
if isinstance(all_shap, list): all_shap = all_shap[1]

top_factors = []
for i in range(len(churn_df)):
    row_shap = pd.Series(all_shap[i], index=feature_cols).abs().sort_values(ascending=False)
    top_3 = ", ".join(row_shap.head(3).index)
    top_factors.append(top_3)

output_df = pd.DataFrame({
    "acc_id": churn_df["acc_id"],
    "churn_probability": np.round(all_probs, 4),
    "churn_risk_level": risk_levels,
    "top_risk_factors": top_factors,
    "days_since_last_payment": churn_df["days_since_last_payment"],
    "days_since_last_activity": churn_df["days_since_last_activity"],
    "total_revenue_all": churn_df["total_revenue_all"]
})

output_path = "churn_predictions.csv"
output_df.to_csv(output_path, index=False)
print(f"🎉 บันทึกผลการทำนายเรียบร้อยที่: {output_path}")
print("📊 สรุปสัดส่วนกลุ่มความเสี่ยง:")
print(output_df["churn_risk_level"].value_counts(normalize=True).apply(lambda x: f'{x:.1%}'))
print("\nตัวอย่างลูกค้าที่มีความเสี่ยงสูงสุด 5 รายแรก:")
print(output_df.sort_values("churn_probability", ascending=False).head(5))